# Module 11 v1 — Controlled Subject/Domain Alignment

**Purpose:** evaluate controlled subject/domain-alignment strategies on the frozen Module 10A Variant A backbone, followed by **3-fold grouped confirmation on both source datasets**.

## Frozen backbone
- 22 EEG channels × 640 samples
- 160 Hz, 8–30 Hz
- multi-scale temporal kernels 15/31/63
- depthwise spatial filtering
- gated fusion
- 2-layer compact EEGCCT
- 128-D embedding
- CE baseline
- subject × class balanced sampling
- training-only temporal interpolation augmentation

## Variants
- **A0:** CE only
- **A1:** CE + stable batch-level feature-statistics alignment
- **A2:** CE + subject-adversarial GRL
- **A3:** CE + class-conditional subject alignment

## Confirmation protocol
- BCI-IV-2a: 3-fold `GroupKFold`, group = subject
- EEGMMIDB: 3-fold `GroupKFold`, group = subject
- no target dataset/subject statistics
- no target training
- no target early stopping
- no target hyperparameter tuning
- no target augmentation
- no subject deletion

**Important:** this module is a source-only development/confirmation experiment. It does **not** produce the final outer LOSO result and does not use target subjects for selection.


In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY
# ============================================================

import os
import gc
import json
import math
import time
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import h5py

from tqdm.auto import tqdm

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
    TensorDataset,
)

warnings.filterwarnings("ignore")

SEED = 20260822

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("=" * 78)
print("MODULE 11 v1")
print("=" * 78)
print("Python-compatible PyTorch device:", DEVICE)
print("Seed:", SEED)


MODULE 11 v1
Python-compatible PyTorch device: mps
Seed: 20260822


In [2]:
# ============================================================
# CELL 2 — PATHS + CONFIGURATION
# ============================================================

PROJECT_ROOT = Path(
    "/Users/ashokvarmabevara/Project2/cross_dataset_mi_project"
)

CACHE_PATH = (
    PROJECT_ROOT
    / "cache"
    / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
)

META_PATH = (
    PROJECT_ROOT
    / "manifests"
    / "module_6_cache_metadata.csv"
)

MODULE11_ROOT = (
    PROJECT_ROOT
    / "results"
    / "module_11_v1_controlled_subject_domain_alignment"
)

CHECKPOINT_ROOT = MODULE11_ROOT / "checkpoints"
HISTORY_ROOT = MODULE11_ROOT / "histories"

MODULE11_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
HISTORY_ROOT.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    # data
    n_channels: int = 22
    n_samples: int = 640
    target_sfreq: float = 160.0
    low_hz: float = 8.0
    high_hz: float = 30.0
    classes: tuple = ("left", "right", "feet")

    # backbone — frozen from Module 10A Variant A
    temporal_width: int = 16
    branch_width: int = 32
    fusion_width: int = 64
    token_dim: int = 96
    patch_size: int = 16
    transformer_heads: int = 4
    transformer_ffn: int = 192
    transformer_layers: int = 2
    embedding_dim: int = 128
    dropout: float = 0.15

    # optimization
    epochs: int = 30
    batch_size: int = 96
    batch_subjects: int = 8
    batches_per_epoch_min: int = 32
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    early_stopping_patience: int = 7

    # alignment
    alignment_projection_dim: int = 32
    alignment_weight: float = 0.02
    grl_max_lambda: float = 0.05
    grl_warmup_epochs: int = 10

    # validation
    n_splits: int = 3

    # execution
    smoke_test: bool = False
    smoke_folds_per_dataset: int = 1
    run_variants = ("A0", "A1", "A2", "A3")
    num_workers: int = 0

CFG = Config()

print(json.dumps(asdict(CFG), indent=2, default=list))


{
  "n_channels": 22,
  "n_samples": 640,
  "target_sfreq": 160.0,
  "low_hz": 8.0,
  "high_hz": 30.0,
  "classes": [
    "left",
    "right",
    "feet"
  ],
  "temporal_width": 16,
  "branch_width": 32,
  "fusion_width": 64,
  "token_dim": 96,
  "patch_size": 16,
  "transformer_heads": 4,
  "transformer_ffn": 192,
  "transformer_layers": 2,
  "embedding_dim": 128,
  "dropout": 0.15,
  "epochs": 30,
  "batch_size": 96,
  "batch_subjects": 8,
  "batches_per_epoch_min": 32,
  "lr": 0.0002,
  "weight_decay": 0.0001,
  "grad_clip": 1.0,
  "early_stopping_patience": 7,
  "alignment_projection_dim": 32,
  "alignment_weight": 0.02,
  "grl_max_lambda": 0.05,
  "grl_warmup_epochs": 10,
  "n_splits": 3,
  "smoke_test": false,
  "smoke_folds_per_dataset": 1,
  "num_workers": 0
}


In [3]:
# ============================================================
# CELL 3 — LOAD CACHE + METADATA ROBUSTLY
# ============================================================

assert CACHE_PATH.exists(), f"Missing cache: {CACHE_PATH}"
assert META_PATH.exists(), f"Missing metadata: {META_PATH}"

meta = pd.read_csv(META_PATH)

with h5py.File(CACHE_PATH, "r") as h5:
    print("HDF5 root keys:")
    for key in h5.keys():
        obj = h5[key]
        kind = "dataset" if isinstance(obj, h5py.Dataset) else "group"
        shape = getattr(obj, "shape", None)
        dtype = getattr(obj, "dtype", None)
        print(f"  {key}: {kind}, shape={shape}, dtype={dtype}")

    assert "X" in h5, "Expected X dataset in cache."
    X_all = h5["X"][:]

print()
print("X shape:", X_all.shape)
print("X dtype:", X_all.dtype)
print("Metadata shape:", meta.shape)
print("Metadata columns:")
print(list(meta.columns))

assert X_all.ndim == 3
assert X_all.shape[1:] == (CFG.n_channels, CFG.n_samples)
assert len(meta) == len(X_all)

print()
print("Datasets:")
print(meta["dataset"].value_counts(dropna=False))

print()
print("Subjects:", meta["subject"].nunique())


HDF5 root keys:
  X: dataset, shape=(9316, 22, 640), dtype=float32
  metadata: group, shape=None, dtype=None

X shape: (9316, 22, 640)
X dtype: float32
Metadata shape: (9316, 11)
Metadata columns:
['cache_index', 'dataset', 'subject', 'run', 'recording_id', 'filename', 'absolute_path', 'harmonized_class', 'event_index', 'onset_sec', 'source_sfreq_hz']

Datasets:
dataset
EEGMMIDB     7372
BCI-IV-2a    1944
Name: count, dtype: int64

Subjects: 118


In [4]:
# ============================================================
# CELL 4 — NORMALIZE METADATA NAMES + PROTOCOL AUDIT
# ============================================================

meta = meta.copy()

# Flexible class-column resolution.
class_candidates = [
    "harmonized_class",
    "class",
    "label",
    "class_name",
]
CLASS_COL = next((c for c in class_candidates if c in meta.columns), None)
assert CLASS_COL is not None, (
    f"Could not find class column. Tried {class_candidates}. "
    f"Available: {list(meta.columns)}"
)

assert "dataset" in meta.columns
assert "subject" in meta.columns

# Stable string representation for grouping and reporting.
meta["subject"] = meta["subject"].astype(str)
meta["dataset"] = meta["dataset"].astype(str)
meta[CLASS_COL] = meta[CLASS_COL].astype(str)

# Map labels to fixed 3-class IDs.
class_to_id = {name: i for i, name in enumerate(CFG.classes)}
meta["class_id"] = meta[CLASS_COL].map(class_to_id)

assert meta["class_id"].notna().all(), (
    "Unmapped classes detected: "
    f"{sorted(meta.loc[meta['class_id'].isna(), CLASS_COL].unique())}"
)
meta["class_id"] = meta["class_id"].astype(int)

# Basic integrity.
assert np.isfinite(X_all).all(), "Cache contains NaN/Inf."
assert meta["class_id"].isin(range(len(CFG.classes))).all()

print("=" * 78)
print("MODULE 11 DATA AUDIT")
print("=" * 78)

for ds in sorted(meta["dataset"].unique()):
    d = meta.loc[meta["dataset"] == ds]
    print()
    print(ds)
    print("  trials   :", len(d))
    print("  subjects :", d["subject"].nunique())
    print("  classes  :", d[CLASS_COL].value_counts().to_dict())

print()
print("Class mapping:", class_to_id)


MODULE 11 DATA AUDIT

BCI-IV-2a
  trials   : 1944
  subjects : 9
  classes  : {'feet': 648, 'right': 648, 'left': 648}

EEGMMIDB
  trials   : 7372
  subjects : 109
  classes  : {'left': 2479, 'feet': 2455, 'right': 2438}

Class mapping: {'left': 0, 'right': 1, 'feet': 2}


In [5]:
# ============================================================
# CELL 5 — SOURCE-ONLY ROBUST NORMALIZER
# ============================================================

class SourceRobustNormalizer:
    def __init__(self, eps: float = 1e-6):
        self.eps = eps
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = set()
        self.fitted_dataset_ = None

    def fit(self, X, subjects, dataset_name):
        X = np.asarray(X, dtype=np.float32)
        subjects = np.asarray(subjects).astype(str)

        self.median_ = np.median(
            X.astype(np.float64),
            axis=(0, 2),
        ).astype(np.float32)

        q75 = np.percentile(
            X.astype(np.float64),
            75,
            axis=(0, 2),
        )
        q25 = np.percentile(
            X.astype(np.float64),
            25,
            axis=(0, 2),
        )

        self.iqr_ = np.maximum(
            (q75 - q25).astype(np.float32),
            self.eps,
        )

        self.fitted_subjects_ = set(subjects.tolist())
        self.fitted_dataset_ = str(dataset_name)

        return self

    def transform(self, X):
        assert self.median_ is not None
        X = np.asarray(X, dtype=np.float32)
        return (
            (X - self.median_[None, :, None])
            / self.iqr_[None, :, None]
        ).astype(np.float32)

    def assert_target_excluded(self, target_subjects):
        overlap = self.fitted_subjects_.intersection(
            set(map(str, target_subjects))
        )
        assert not overlap, (
            f"Normalizer leakage: target subjects present in fit: "
            f"{sorted(overlap)}"
        )

print("SourceRobustNormalizer ready.")


SourceRobustNormalizer ready.


In [6]:
# ============================================================
# CELL 6 — TRAINING-ONLY TEMPORAL INTERPOLATION AUGMENTATION
# ============================================================

def temporal_interpolation_crop(
    x: torch.Tensor,
    output_len: int,
    crop_min_fraction: float = 0.80,
) -> torch.Tensor:
    # x: [C, T]
    C, T = x.shape

    min_len = int(round(T * crop_min_fraction))
    crop_len = random.randint(min_len, T)

    if crop_len == T:
        return x

    start = random.randint(0, T - crop_len)
    crop = x[:, start:start + crop_len]

    crop = crop.unsqueeze(0)  # [1, C, L]
    crop = F.interpolate(
        crop,
        size=output_len,
        mode="linear",
        align_corners=False,
    )
    return crop.squeeze(0)


def training_augmentation(
    x: torch.Tensor,
    output_len: int,
) -> torch.Tensor:

    # Temporal crop on ~65% of training samples.
    if random.random() < 0.65:
        x = temporal_interpolation_crop(
            x,
            output_len=output_len,
            crop_min_fraction=0.80,
        )

    # Small amplitude scaling.
    if random.random() < 0.35:
        scale = torch.empty(
            1,
            device=x.device,
        ).uniform_(0.90, 1.10)
        x = x * scale

    # Small additive noise.
    if random.random() < 0.25:
        noise_std = (
            0.01
            * x.std(
                dim=-1,
                keepdim=True,
            ).clamp_min(1e-5)
        )
        x = x + torch.randn_like(x) * noise_std

    # Mild time masking.
    if random.random() < 0.20:
        width = max(4, output_len // 24)
        start = random.randint(
            0,
            max(0, output_len - width),
        )
        x[:, start:start + width] = 0.0

    return x


# Deterministic shape check.
tmp = torch.zeros(CFG.n_channels, CFG.n_samples)
aug = training_augmentation(tmp, CFG.n_samples)
assert aug.shape == tmp.shape

print("Temporal augmentation smoke test: PASS")


Temporal augmentation smoke test: PASS


In [7]:
# ============================================================
# CELL 7 — SUBJECT × CLASS BALANCED BATCH SAMPLER
# ============================================================

class SubjectClassBalancedBatchSampler(Sampler):
    def __init__(
        self,
        subject_array,
        class_array,
        batch_subjects,
        batches_per_epoch,
        seed=SEED,
    ):
        self.subject_array = np.asarray(subject_array).astype(str)
        self.class_array = np.asarray(class_array).astype(int)
        self.batch_subjects = int(batch_subjects)
        self.batches_per_epoch = int(batches_per_epoch)
        self.seed = int(seed)
        self.epoch = 0

        self.subjects = np.unique(self.subject_array).tolist()

        self.by_subject_class = {}
        for s in self.subjects:
            for c in range(len(CFG.classes)):
                idx = np.flatnonzero(
                    (self.subject_array == s)
                    & (self.class_array == c)
                )
                assert len(idx) > 0, (
                    f"Missing class {c} for subject {s}"
                )
                self.by_subject_class[(s, c)] = idx

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(
            self.seed + self.epoch
        )

        for _ in range(self.batches_per_epoch):
            if len(self.subjects) <= self.batch_subjects:
                chosen_subjects = list(self.subjects)
            else:
                chosen_subjects = rng.choice(
                    self.subjects,
                    size=self.batch_subjects,
                    replace=False,
                ).tolist()

            batch = []

            for s in chosen_subjects:
                for c in range(len(CFG.classes)):
                    pool = self.by_subject_class[(s, c)]
                    batch.append(
                        int(
                            rng.choice(
                                pool,
                                size=1,
                                replace=False,
                            )[0]
                        )
                    )

            rng.shuffle(batch)
            yield batch

    def __len__(self):
        return self.batches_per_epoch


def build_batch_sampler(
    subjects,
    classes,
    n_samples,
):
    n_subjects = len(np.unique(subjects))
    batches = max(
        CFG.batches_per_epoch_min,
        int(
            math.ceil(
                n_samples / CFG.batch_size
            )
        ),
    )
    batches = max(
        batches,
        8 if n_subjects < CFG.batch_subjects else 1,
    )
    return SubjectClassBalancedBatchSampler(
        subject_array=subjects,
        class_array=classes,
        batch_subjects=min(
            CFG.batch_subjects,
            n_subjects,
        ),
        batches_per_epoch=batches,
    )

print("Balanced sampler ready.")


Balanced sampler ready.


In [8]:
# ============================================================
# CELL 8 — DATASET / LOADER
# ============================================================

class EEGDataset(Dataset):
    def __init__(
        self,
        X,
        y,
        subjects,
        training=False,
    ):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subjects = np.asarray(subjects).astype(str)
        self.training = bool(training)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx].copy())
        y = torch.tensor(
            int(self.y[idx]),
            dtype=torch.long,
        )

        if self.training:
            x = training_augmentation(
                x,
                CFG.n_samples,
            )

        return {
            "x": x,
            "y": y,
            "subject": self.subjects[idx],
        }


def make_loader(
    X,
    y,
    subjects,
    training,
):
    ds = EEGDataset(
        X,
        y,
        subjects,
        training=training,
    )

    if training:
        sampler = build_batch_sampler(
            subjects,
            y,
            len(y),
        )
        return DataLoader(
            ds,
            batch_sampler=sampler,
            num_workers=CFG.num_workers,
            pin_memory=False,
        )

    return DataLoader(
        ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=False,
    )

print("Dataset/loader ready.")


Dataset/loader ready.


In [9]:
# ============================================================
# CELL 9 — FROZEN MODULE 10A VARIANT A BACKBONE
# ============================================================

class TemporalSpatialBranch(nn.Module):
    def __init__(
        self,
        width=16,
        branch_width=32,
        kernel=15,
        dropout=0.15,
    ):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(
                1,
                width,
                kernel_size=(1, kernel),
                padding=(0, kernel // 2),
                bias=False,
            ),
            nn.BatchNorm2d(width),
            nn.ELU(),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(
                width,
                width,
                kernel_size=(CFG.n_channels, 1),
                groups=width,
                bias=False,
            ),
            nn.BatchNorm2d(width),
            nn.ELU(),

            nn.Conv2d(
                width,
                branch_width,
                kernel_size=(1, 3),
                padding=(0, 1),
                bias=False,
            ),
            nn.BatchNorm2d(branch_width),
            nn.ELU(),

            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.spatial(self.temporal(x))


class FrozenVariantABackbone(nn.Module):
    def __init__(self):
        super().__init__()

        kernels = (15, 31, 63)

        self.branches = nn.ModuleList([
            TemporalSpatialBranch(
                width=CFG.temporal_width,
                branch_width=CFG.branch_width,
                kernel=k,
                dropout=CFG.dropout,
            )
            for k in kernels
        ])

        cat_width = len(kernels) * CFG.branch_width

        self.fusion_proj = nn.Conv2d(
            cat_width,
            CFG.fusion_width,
            kernel_size=1,
            bias=False,
        )

        self.fusion_bn = nn.BatchNorm2d(
            CFG.fusion_width
        )

        self.fusion_gate = nn.Conv2d(
            cat_width,
            CFG.fusion_width,
            kernel_size=1,
            bias=True,
        )

        self.tokenizer = nn.Sequential(
            nn.Conv2d(
                CFG.fusion_width,
                CFG.token_dim,
                kernel_size=(1, CFG.patch_size),
                stride=(1, CFG.patch_size),
                bias=False,
            ),
            nn.BatchNorm2d(CFG.token_dim),
            nn.ELU(),
        )

        n_tokens = CFG.n_samples // CFG.patch_size

        enc = nn.TransformerEncoderLayer(
            d_model=CFG.token_dim,
            nhead=CFG.transformer_heads,
            dim_feedforward=CFG.transformer_ffn,
            dropout=CFG.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            enc,
            num_layers=CFG.transformer_layers,
        )

        self.positional = nn.Parameter(
            torch.zeros(
                1,
                n_tokens,
                CFG.token_dim,
            )
        )
        nn.init.normal_(
            self.positional,
            std=0.02,
        )

        self.attn = nn.Sequential(
            nn.Linear(
                CFG.token_dim,
                CFG.token_dim // 2,
            ),
            nn.Tanh(),
            nn.Linear(
                CFG.token_dim // 2,
                1,
            ),
        )

        self.embedding = nn.Sequential(
            nn.Linear(
                CFG.token_dim,
                CFG.embedding_dim,
            ),
            nn.LayerNorm(
                CFG.embedding_dim,
            ),
            nn.ELU(),
            nn.Dropout(0.20),
        )

        self.classifier = nn.Linear(
            CFG.embedding_dim,
            len(CFG.classes),
        )

        self.alignment_projection = nn.Linear(
            CFG.fusion_width,
            CFG.alignment_projection_dim,
            bias=False,
        )

    def forward(self, x):
        # x = [B, C, T]
        x = x.unsqueeze(1)

        cat = torch.cat(
            [branch(x) for branch in self.branches],
            dim=1,
        )

        proj = self.fusion_bn(
            self.fusion_proj(cat)
        )

        gate = torch.sigmoid(
            self.fusion_gate(cat)
        )

        fused = F.elu(
            proj * gate
        )

        # Batch-level alignment representation.
        base_alignment = (
            fused
            .squeeze(2)
            .mean(dim=-1)
        )

        alignment_feature = (
            self.alignment_projection(
                base_alignment
            )
        )

        tokens = self.tokenizer(
            fused
        ).squeeze(2)

        tokens = tokens.transpose(1, 2)

        tokens = (
            tokens
            + self.positional[
                :, :tokens.shape[1]
            ]
        )

        tokens = self.encoder(tokens)

        scores = self.attn(
            tokens
        ).squeeze(-1)

        weights = torch.softmax(
            scores,
            dim=1,
        )

        pooled = torch.sum(
            tokens
            * weights.unsqueeze(-1),
            dim=1,
        )

        embedding = self.embedding(
            pooled
        )

        logits = self.classifier(
            embedding
        )

        return {
            "logits": logits,
            "embedding": embedding,
            "alignment_feature": alignment_feature,
            "attention": weights,
        }


def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

model_smoke = FrozenVariantABackbone().to(DEVICE)
with torch.no_grad():
    dummy = torch.zeros(
        4,
        CFG.n_channels,
        CFG.n_samples,
        device=DEVICE,
    )
    test_out = model_smoke(dummy)

assert test_out["logits"].shape == (4, 3)
assert test_out["embedding"].shape == (
    4,
    CFG.embedding_dim,
)
assert test_out["alignment_feature"].shape == (
    4,
    CFG.alignment_projection_dim,
)

print(
    "Parameters:",
    f"{count_trainable_parameters(model_smoke):,}",
)
print(
    "Shape smoke test: PASS"
)

del model_smoke, dummy, test_out
gc.collect()
if DEVICE.type == "mps":
    torch.mps.empty_cache()


Parameters: 291,988
Shape smoke test: PASS


In [10]:
# ============================================================
# CELL 10 — ALIGNMENT LOSSES
# ============================================================

def pairwise_mean_alignment(
    features,
    subject_ids,
):
    # Stable mean-statistics alignment.
    # Each subject needs >= 1 observation.
    subjects = np.asarray(subject_ids).astype(str)

    groups = []
    for s in np.unique(subjects):
        mask = torch.tensor(
            subjects == s,
            device=features.device,
            dtype=torch.bool,
        )
        if int(mask.sum().item()) < 1:
            continue
        groups.append(
            features[mask].mean(dim=0)
        )

    if len(groups) < 2:
        return features.new_zeros(())

    G = torch.stack(groups, dim=0)

    global_mean = G.mean(dim=0, keepdim=True)
    return (
        (G - global_mean)
        .pow(2)
        .mean()
    )


def class_conditional_alignment(
    features,
    class_ids,
    subject_ids,
):
    classes = np.asarray(class_ids).astype(int)
    subjects = np.asarray(subject_ids).astype(str)

    losses = []

    for c in range(len(CFG.classes)):
        c_subject_means = []

        for s in np.unique(subjects):
            mask_np = (
                (classes == c)
                & (subjects == s)
            )

            if not np.any(mask_np):
                continue

            mask = torch.tensor(
                mask_np,
                device=features.device,
                dtype=torch.bool,
            )

            c_subject_means.append(
                features[mask].mean(dim=0)
            )

        if len(c_subject_means) < 2:
            continue

        G = torch.stack(
            c_subject_means,
            dim=0,
        )

        global_mean = G.mean(
            dim=0,
            keepdim=True,
        )

        losses.append(
            (G - global_mean)
            .pow(2)
            .mean()
        )

    if not losses:
        return features.new_zeros(())

    return torch.stack(losses).mean()


class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = float(lambda_)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return (
            -ctx.lambda_ * grad_output,
            None,
        )


def grad_reverse(x, lambda_):
    return GradientReversalFn.apply(
        x,
        lambda_,
    )


def grl_lambda(
    epoch,
    max_lambda,
    warmup_epochs,
):
    if warmup_epochs <= 0:
        return max_lambda

    progress = min(
        1.0,
        float(epoch + 1)
        / float(warmup_epochs),
    )
    return max_lambda * progress


print("Alignment losses ready.")


Alignment losses ready.


In [11]:
# ============================================================
# CELL 11 — SUBJECT ADVERSARIAL HEAD
# ============================================================

class SubjectAdversarialHead(nn.Module):
    def __init__(self, n_subjects):
        super().__init__()

        hidden = 96

        self.net = nn.Sequential(
            nn.Linear(
                CFG.embedding_dim,
                hidden,
            ),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(
                hidden,
                n_subjects,
            ),
        )

    def forward(
        self,
        embedding,
        grl_strength,
    ):
        z = grad_reverse(
            embedding,
            grl_strength,
        )
        return self.net(z)


print("Subject adversarial head ready.")


Subject adversarial head ready.


In [12]:
# ============================================================
# CELL 12 — TRAIN/VALIDATION UTILITIES
# ============================================================

def make_optimizer(model):
    return torch.optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay,
    )


def move_batch_to_device(batch):
    x = batch["x"].to(
        DEVICE,
        non_blocking=True,
    )
    y = batch["y"].to(
        DEVICE,
        non_blocking=True,
    )
    subjects = np.asarray(
        batch["subject"]
    ).astype(str)
    return x, y, subjects


@torch.no_grad()
def evaluate_model(
    model,
    loader,
):
    model.eval()

    ys = []
    preds = []
    embeddings = []
    subjects = []

    for batch in loader:
        x, y, s = move_batch_to_device(batch)

        out = model(x)

        pred = (
            out["logits"]
            .argmax(dim=1)
            .detach()
            .cpu()
            .numpy()
        )

        ys.extend(
            y.detach()
            .cpu()
            .numpy()
            .tolist()
        )
        preds.extend(pred.tolist())
        embeddings.append(
            out["embedding"]
            .detach()
            .cpu()
        )
        subjects.extend(s.tolist())

    y_true = np.asarray(ys, dtype=int)
    y_pred = np.asarray(preds, dtype=int)

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "kappa": float(
            cohen_kappa_score(
                y_true,
                y_pred,
            )
        ),
        "y_true": y_true,
        "y_pred": y_pred,
        "subjects": np.asarray(subjects),
        "embeddings": torch.cat(
            embeddings,
            dim=0,
        ).numpy(),
    }


def save_checkpoint(
    path,
    model,
    optimizer,
    epoch,
    metrics,
    variant,
    dataset,
):
    torch.save(
        {
            "epoch": int(epoch),
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "metrics": metrics,
            "variant": variant,
            "dataset": dataset,
            "config": asdict(CFG),
        },
        path,
    )


print("Training utilities ready.")


Training utilities ready.


In [13]:
# ============================================================
# CELL 13 — SINGLE FOLD TRAINING
# ============================================================

def train_one_fold(
    X_train,
    y_train,
    s_train,
    X_val,
    y_val,
    s_val,
    variant,
    dataset_name,
    fold_id,
):
    # --------------------------------------------------------
    # Fit source-only normalizer using training subjects only.
    # --------------------------------------------------------
    normalizer = SourceRobustNormalizer()

    normalizer.fit(
        X_train,
        s_train,
        dataset_name,
    )

    # Validation subjects must not enter normalizer fit.
    normalizer.assert_target_excluded(
        np.unique(s_val)
    )

    X_train_n = normalizer.transform(X_train)
    X_val_n = normalizer.transform(X_val)

    train_loader = make_loader(
        X_train_n,
        y_train,
        s_train,
        training=True,
    )

    val_loader = make_loader(
        X_val_n,
        y_val,
        s_val,
        training=False,
    )

    model = FrozenVariantABackbone().to(
        DEVICE
    )

    subject_values = sorted(
        np.unique(s_train).tolist()
    )
    subject_to_id = {
        s: i
        for i, s in enumerate(subject_values)
    }

    adv_head = None
    if variant == "A2":
        adv_head = SubjectAdversarialHead(
            n_subjects=len(subject_values)
        ).to(DEVICE)

    if adv_head is None:
        optimizer = make_optimizer(model)
    else:
        optimizer = torch.optim.AdamW(
            list(model.parameters())
            + list(adv_head.parameters()),
            lr=CFG.lr,
            weight_decay=CFG.weight_decay,
        )

    ce_loss_fn = nn.CrossEntropyLoss()

    best_state = None
    best_metrics = None
    best_epoch = None
    patience_count = 0
    history = []

    for epoch in range(CFG.epochs):
        model.train()
        if adv_head is not None:
            adv_head.train()

        if hasattr(
            train_loader,
            "batch_sampler",
        ) and hasattr(
            train_loader.batch_sampler,
            "set_epoch",
        ):
            train_loader.batch_sampler.set_epoch(
                epoch
            )

        running = {
            "total_loss": 0.0,
            "class_loss": 0.0,
            "alignment_loss": 0.0,
            "domain_loss": 0.0,
            "steps": 0,
        }

        current_grl = grl_lambda(
            epoch,
            CFG.grl_max_lambda,
            CFG.grl_warmup_epochs,
        )

        for batch in train_loader:
            x, y, s = move_batch_to_device(batch)

            optimizer.zero_grad(set_to_none=True)

            out = model(x)

            class_loss = ce_loss_fn(
                out["logits"],
                y,
            )

            alignment_loss = (
                x.new_zeros(())
            )
            domain_loss = (
                x.new_zeros(())
            )

            if variant == "A1":
                alignment_loss = (
                    pairwise_mean_alignment(
                        out["alignment_feature"],
                        s,
                    )
                )

            elif variant == "A2":
                domain_targets = torch.tensor(
                    [
                        subject_to_id[str(v)]
                        for v in s
                    ],
                    dtype=torch.long,
                    device=DEVICE,
                )

                domain_logits = adv_head(
                    out["embedding"],
                    current_grl,
                )

                domain_loss = ce_loss_fn(
                    domain_logits,
                    domain_targets,
                )

            elif variant == "A3":
                alignment_loss = (
                    class_conditional_alignment(
                        out["alignment_feature"],
                        y.detach()
                        .cpu()
                        .numpy(),
                        s,
                    )
                )

            if variant in {"A1", "A3"}:
                total_loss = (
                    class_loss
                    + CFG.alignment_weight
                    * alignment_loss
                )

            elif variant == "A2":
                total_loss = (
                    class_loss
                    + domain_loss
                )

            else:
                total_loss = class_loss

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                list(model.parameters())
                + (
                    list(adv_head.parameters())
                    if adv_head is not None
                    else []
                ),
                CFG.grad_clip,
            )

            optimizer.step()

            running["total_loss"] += float(
                total_loss.detach().cpu()
            )
            running["class_loss"] += float(
                class_loss.detach().cpu()
            )
            running["alignment_loss"] += float(
                alignment_loss.detach().cpu()
            )
            running["domain_loss"] += float(
                domain_loss.detach().cpu()
            )
            running["steps"] += 1

        val_metrics = evaluate_model(
            model,
            val_loader,
        )

        row = {
            "epoch": epoch + 1,
            "train_total_loss": (
                running["total_loss"]
                / max(1, running["steps"])
            ),
            "train_class_loss": (
                running["class_loss"]
                / max(1, running["steps"])
            ),
            "train_alignment_loss": (
                running["alignment_loss"]
                / max(1, running["steps"])
            ),
            "train_domain_loss": (
                running["domain_loss"]
                / max(1, running["steps"])
            ),
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": (
                val_metrics["balanced_accuracy"]
            ),
            "val_macro_f1": val_metrics["macro_f1"],
            "val_kappa": val_metrics["kappa"],
            "grl_lambda": current_grl,
        }

        history.append(row)

        score = (
            val_metrics["balanced_accuracy"],
            val_metrics["macro_f1"],
        )

        if (
            best_metrics is None
            or score
            > (
                best_metrics["balanced_accuracy"],
                best_metrics["macro_f1"],
            )
        ):
            best_metrics = {
                "accuracy": val_metrics["accuracy"],
                "balanced_accuracy": val_metrics[
                    "balanced_accuracy"
                ],
                "macro_f1": val_metrics[
                    "macro_f1"
                ],
                "kappa": val_metrics["kappa"],
            }
            best_epoch = epoch + 1
            best_state = {
                "model": {
                    k: v.detach()
                    .cpu()
                    .clone()
                    for k, v in model.state_dict().items()
                },
                "adv": (
                    {
                        k: v.detach()
                        .cpu()
                        .clone()
                        for k, v in adv_head.state_dict().items()
                    }
                    if adv_head is not None
                    else None
                ),
            }
            patience_count = 0
        else:
            patience_count += 1

        print(
            f"[{dataset_name}][{variant}] "
            f"fold={fold_id} "
            f"ep={epoch+1:02d} "
            f"loss={row['train_total_loss']:.4f} "
            f"valBA={row['val_balanced_accuracy']:.4f} "
            f"valF1={row['val_macro_f1']:.4f}"
        )

        if patience_count >= CFG.early_stopping_patience:
            print(
                f"Early stopping at epoch {epoch+1}"
            )
            break

    # Restore best model.
    if best_state is not None:
        model.load_state_dict(
            best_state["model"]
        )
        if adv_head is not None:
            adv_head.load_state_dict(
                best_state["adv"]
            )

    # Final best-state validation artifacts.
    best_val = evaluate_model(
        model,
        val_loader,
    )

    ckpt_path = (
        CHECKPOINT_ROOT
        / f"{dataset_name}_{variant}_fold{fold_id}.pt"
    )

    save_checkpoint(
        ckpt_path,
        model,
        optimizer,
        best_epoch,
        best_metrics,
        variant,
        dataset_name,
    )

    history_df = pd.DataFrame(history)
    history_path = (
        HISTORY_ROOT
        / f"{dataset_name}_{variant}_fold{fold_id}.csv"
    )
    history_df.to_csv(
        history_path,
        index=False,
    )

    result = {
        "dataset": dataset_name,
        "variant": variant,
        "fold": int(fold_id),
        "best_epoch": int(best_epoch),
        "accuracy": best_val["accuracy"],
        "balanced_accuracy": best_val[
            "balanced_accuracy"
        ],
        "macro_f1": best_val["macro_f1"],
        "kappa": best_val["kappa"],
        "n_train_subjects": int(
            len(np.unique(s_train))
        ),
        "n_val_subjects": int(
            len(np.unique(s_val))
        ),
        "train_subjects": "|".join(
            sorted(np.unique(s_train))
        ),
        "val_subjects": "|".join(
            sorted(np.unique(s_val))
        ),
        "normalizer_subjects": "|".join(
            sorted(normalizer.fitted_subjects_)
        ),
        "target_excluded": True,
        "checkpoint": str(ckpt_path),
        "history_path": str(history_path),
    }

    # Free memory.
    del (
        model,
        adv_head,
        optimizer,
        train_loader,
        val_loader,
        normalizer,
    )
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    return result


In [14]:
# ============================================================
# CELL 14 — GROUPED 3-FOLD CONFIRMATION RUNNER
# ============================================================

def run_dataset_confirmation(dataset_name):
    d = meta.loc[
        meta["dataset"] == dataset_name
    ].copy()

    assert d["subject"].nunique() >= CFG.n_splits, (
        f"{dataset_name} has only "
        f"{d['subject'].nunique()} subjects; "
        f"cannot perform {CFG.n_splits}-fold GroupKFold."
    )

    idx = d.index.to_numpy()
    groups = d["subject"].to_numpy()

    splitter = GroupKFold(
        n_splits=CFG.n_splits
    )

    results = []

    split_iter = splitter.split(
        idx,
        d["class_id"].to_numpy(),
        groups,
    )

    if CFG.smoke_test:
        selected_splits = []
        for j, split in enumerate(split_iter):
            selected_splits.append(split)
            if (
                j + 1
                >= CFG.smoke_folds_per_dataset
            ):
                break
    else:
        selected_splits = list(split_iter)

    for variant in CFG.run_variants:

        print()
        print("=" * 78)
        print(
            f"{dataset_name} | {variant} "
            f"| {len(selected_splits)} fold(s)"
        )
        print("=" * 78)

        for fold_id, (train_pos, val_pos) in enumerate(
            tqdm(
                selected_splits,
                desc=f"{dataset_name} {variant}",
            ),
            start=1,
        ):
            train_idx = idx[train_pos]
            val_idx = idx[val_pos]

            train_df = meta.loc[
                train_idx
            ].copy()
            val_df = meta.loc[
                val_idx
            ].copy()

            # Hard grouped split guard.
            overlap = (
                set(train_df["subject"])
                & set(val_df["subject"])
            )
            assert not overlap, (
                f"Group leakage in fold {fold_id}: "
                f"{sorted(overlap)}"
            )

            # Extract X by positional row number in full cache.
            X_train = X_all[
                train_df.index.to_numpy()
            ]
            X_val = X_all[
                val_df.index.to_numpy()
            ]

            result = train_one_fold(
                X_train=X_train,
                y_train=train_df["class_id"].to_numpy(),
                s_train=train_df["subject"].to_numpy(),
                X_val=X_val,
                y_val=val_df["class_id"].to_numpy(),
                s_val=val_df["subject"].to_numpy(),
                variant=variant,
                dataset_name=dataset_name,
                fold_id=fold_id,
            )

            results.append(result)

    return results


dataset_names = [
    "BCI-IV-2a",
    "EEGMMIDB",
]

all_results = []

for dataset_name in dataset_names:
    ds_results = run_dataset_confirmation(
        dataset_name
    )
    all_results.extend(ds_results)

results_df = pd.DataFrame(all_results)

print()
print("=" * 78)
print("MODULE 11 RUN COMPLETE")
print("=" * 78)
display(
    results_df[
        [
            "dataset",
            "variant",
            "fold",
            "best_epoch",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "kappa",
            "n_train_subjects",
            "n_val_subjects",
        ]
    ]
)



BCI-IV-2a | A0 | 3 fold(s)


BCI-IV-2a A0:   0%|          | 0/3 [00:00<?, ?it/s]

[BCI-IV-2a][A0] fold=1 ep=01 loss=1.1105 valBA=0.4213 valF1=0.4127
[BCI-IV-2a][A0] fold=1 ep=02 loss=1.0486 valBA=0.4938 valF1=0.4809
[BCI-IV-2a][A0] fold=1 ep=03 loss=0.9758 valBA=0.5386 valF1=0.5439
[BCI-IV-2a][A0] fold=1 ep=04 loss=0.9685 valBA=0.5494 valF1=0.5410
[BCI-IV-2a][A0] fold=1 ep=05 loss=0.9791 valBA=0.5463 valF1=0.5499
[BCI-IV-2a][A0] fold=1 ep=06 loss=0.9261 valBA=0.5309 valF1=0.5285
[BCI-IV-2a][A0] fold=1 ep=07 loss=0.9339 valBA=0.5386 valF1=0.5196
[BCI-IV-2a][A0] fold=1 ep=08 loss=0.8174 valBA=0.5864 valF1=0.5904
[BCI-IV-2a][A0] fold=1 ep=09 loss=0.8706 valBA=0.5478 valF1=0.5393
[BCI-IV-2a][A0] fold=1 ep=10 loss=0.8771 valBA=0.5895 valF1=0.5909
[BCI-IV-2a][A0] fold=1 ep=11 loss=0.8140 valBA=0.5617 valF1=0.5545
[BCI-IV-2a][A0] fold=1 ep=12 loss=0.8025 valBA=0.5340 valF1=0.5122
[BCI-IV-2a][A0] fold=1 ep=13 loss=0.8273 valBA=0.5818 valF1=0.5799
[BCI-IV-2a][A0] fold=1 ep=14 loss=0.7729 valBA=0.5278 valF1=0.4967
[BCI-IV-2a][A0] fold=1 ep=15 loss=0.7546 valBA=0.5247 valF1=0.

BCI-IV-2a A1:   0%|          | 0/3 [00:00<?, ?it/s]

[BCI-IV-2a][A1] fold=1 ep=01 loss=1.1323 valBA=0.3889 valF1=0.3779
[BCI-IV-2a][A1] fold=1 ep=02 loss=1.0943 valBA=0.4352 valF1=0.4146
[BCI-IV-2a][A1] fold=1 ep=03 loss=1.0168 valBA=0.4892 valF1=0.4673
[BCI-IV-2a][A1] fold=1 ep=04 loss=1.0249 valBA=0.5077 valF1=0.4702
[BCI-IV-2a][A1] fold=1 ep=05 loss=0.9698 valBA=0.4907 valF1=0.4537
[BCI-IV-2a][A1] fold=1 ep=06 loss=0.8917 valBA=0.5401 valF1=0.5223
[BCI-IV-2a][A1] fold=1 ep=07 loss=0.9282 valBA=0.5772 valF1=0.5775
[BCI-IV-2a][A1] fold=1 ep=08 loss=0.8562 valBA=0.5540 valF1=0.5495
[BCI-IV-2a][A1] fold=1 ep=09 loss=0.8763 valBA=0.5664 valF1=0.5605
[BCI-IV-2a][A1] fold=1 ep=10 loss=0.8555 valBA=0.5401 valF1=0.5296
[BCI-IV-2a][A1] fold=1 ep=11 loss=0.8463 valBA=0.5741 valF1=0.5636
[BCI-IV-2a][A1] fold=1 ep=12 loss=0.8340 valBA=0.5540 valF1=0.5347
[BCI-IV-2a][A1] fold=1 ep=13 loss=0.8033 valBA=0.5679 valF1=0.5530
[BCI-IV-2a][A1] fold=1 ep=14 loss=0.7766 valBA=0.5710 valF1=0.5555
Early stopping at epoch 14
[BCI-IV-2a][A1] fold=2 ep=01 loss=1

...

EEGMMIDB A2:   0%|          | 0/3 [00:00<?, ?it/s]

[EEGMMIDB][A2] fold=1 ep=01 loss=5.4282 valBA=0.3636 valF1=0.3600
[EEGMMIDB][A2] fold=1 ep=02 loss=5.3089 valBA=0.3678 valF1=0.3658
[EEGMMIDB][A2] fold=1 ep=03 loss=5.2551 valBA=0.3703 valF1=0.3682
[EEGMMIDB][A2] fold=1 ep=04 loss=5.2068 valBA=0.3989 valF1=0.3865
[EEGMMIDB][A2] fold=1 ep=05 loss=5.1570 valBA=0.4192 valF1=0.4114
[EEGMMIDB][A2] fold=1 ep=06 loss=5.1203 valBA=0.4261 valF1=0.4232
[EEGMMIDB][A2] fold=1 ep=07 loss=5.3299 valBA=0.4304 valF1=0.4278
[EEGMMIDB][A2] fold=1 ep=08 loss=5.5775 valBA=0.4347 valF1=0.4341
[EEGMMIDB][A2] fold=1 ep=09 loss=5.4352 valBA=0.4484 valF1=0.4431
[EEGMMIDB][A2] fold=1 ep=10 loss=5.3631 valBA=0.4330 valF1=0.4245
[EEGMMIDB][A2] fold=1 ep=11 loss=5.3253 valBA=0.4599 valF1=0.4583
[EEGMMIDB][A2] fold=1 ep=12 loss=5.2945 valBA=0.4616 valF1=0.4588
[EEGMMIDB][A2] fold=1 ep=13 loss=5.3233 valBA=0.4703 valF1=0.4701
[EEGMMIDB][A2] fold=1 ep=14 loss=5.3133 valBA=0.4595 valF1=0.4574
[EEGMMIDB][A2] fold=1 ep=15 loss=5.2603 valBA=0.4595 valF1=0.4586
[EEGMMIDB]

EEGMMIDB A3:   0%|          | 0/3 [00:00<?, ?it/s]

[EEGMMIDB][A3] fold=1 ep=01 loss=1.1235 valBA=0.3753 valF1=0.3748
[EEGMMIDB][A3] fold=1 ep=02 loss=1.1022 valBA=0.3615 valF1=0.3494
[EEGMMIDB][A3] fold=1 ep=03 loss=1.0960 valBA=0.3809 valF1=0.3721
[EEGMMIDB][A3] fold=1 ep=04 loss=1.0836 valBA=0.4034 valF1=0.4019
[EEGMMIDB][A3] fold=1 ep=05 loss=1.0743 valBA=0.4229 valF1=0.4229
[EEGMMIDB][A3] fold=1 ep=06 loss=1.0494 valBA=0.4295 valF1=0.4279
[EEGMMIDB][A3] fold=1 ep=07 loss=1.0419 valBA=0.4286 valF1=0.4255
[EEGMMIDB][A3] fold=1 ep=08 loss=1.0370 valBA=0.4337 valF1=0.4327
[EEGMMIDB][A3] fold=1 ep=09 loss=1.0276 valBA=0.4482 valF1=0.4468
[EEGMMIDB][A3] fold=1 ep=10 loss=1.0258 valBA=0.4420 valF1=0.4367
[EEGMMIDB][A3] fold=1 ep=11 loss=1.0411 valBA=0.4448 valF1=0.4434
[EEGMMIDB][A3] fold=1 ep=12 loss=1.0079 valBA=0.4559 valF1=0.4512
[EEGMMIDB][A3] fold=1 ep=13 loss=1.0104 valBA=0.4563 valF1=0.4549
[EEGMMIDB][A3] fold=1 ep=14 loss=1.0177 valBA=0.4656 valF1=0.4625
[EEGMMIDB][A3] fold=1 ep=15 loss=1.0138 valBA=0.4529 valF1=0.4528
[EEGMMIDB]

,dataset,variant,fold,best_epoch,accuracy,balanced_accuracy,macro_f1,kappa,n_train_subjects,n_val_subjects
0,BCI-IV-2a,A0,1,10,0.589506,0.589506,0.590855,0.384259,6,3
1,BCI-IV-2a,A0,2,10,0.504630,0.504630,0.488307,0.256944,6,3
2,BCI-IV-2a,A0,3,5,0.466049,0.466049,0.455395,0.199074,6,3
3,BCI-IV-2a,A1,1,7,0.577160,0.577160,0.577474,0.365741,6,3
4,BCI-IV-2a,A1,2,24,0.512346,0.512346,0.488224,0.268519,6,3
5,BCI-IV-2a,A1,3,10,0.490741,0.490741,0.476809,0.236111,6,3
6,BCI-IV-2a,A2,1,7,0.587963,0.587963,0.585502,0.381944,6,3
7,BCI-IV-2a,A2,2,23,0.503086,0.503086,0.481102,0.254630,6,3
8,BCI-IV-2a,A2,3,6,0.501543,0.501543,0.501540,0.252315,6,3
9,BCI-IV-2a,A3,1,10,0.535494,0.535494,0.511938,0.303241,6,3


In [ ]:
# ============================================================
# CELL 15 — SUBJECT-WISE VALIDATION DIAGNOSTICS
# ============================================================

def compute_subject_metrics_for_checkpoint(
    row
):
    dataset_name = row["dataset"]
    variant = row["variant"]
    fold = int(row["fold"])

    d = meta.loc[
        meta["dataset"] == dataset_name
    ].copy()

    gkf = GroupKFold(
        n_splits=CFG.n_splits
    )

    splits = list(
        gkf.split(
            d.index.to_numpy(),
            d["class_id"].to_numpy(),
            d["subject"].to_numpy(),
        )
    )

    train_pos, val_pos = splits[fold - 1]

    idx = d.index.to_numpy()

    train_idx = idx[train_pos]
    val_idx = idx[val_pos]

    train_df = meta.loc[
        train_idx
    ].copy()
    val_df = meta.loc[
        val_idx
    ].copy()

    normalizer = SourceRobustNormalizer()
    normalizer.fit(
        X_all[train_idx],
        train_df["subject"].to_numpy(),
        dataset_name,
    )
    normalizer.assert_target_excluded(
        val_df["subject"].to_numpy()
    )

    X_val = normalizer.transform(
        X_all[val_idx]
    )

    loader = make_loader(
        X_val,
        val_df["class_id"].to_numpy(),
        val_df["subject"].to_numpy(),
        training=False,
    )

    model = FrozenVariantABackbone().to(
        DEVICE
    )

    checkpoint = torch.load(
        row["checkpoint"],
        map_location="cpu",
    )

    model.load_state_dict(
        checkpoint["model_state"]
    )

    model.eval()

    outputs = []

    with torch.no_grad():
        for batch in loader:
            x, y, subjects = move_batch_to_device(
                batch
            )

            logits = model(x)["logits"]

            pred = logits.argmax(
                dim=1
            ).cpu().numpy()

            outputs.extend(
                zip(
                    subjects.tolist(),
                    y.cpu().numpy().tolist(),
                    pred.tolist(),
                )
            )

    model_subject_df = pd.DataFrame(
        outputs,
        columns=[
            "subject",
            "true_class",
            "pred_class",
        ],
    )

    rows = []

    for subject, sd in model_subject_df.groupby(
        "subject"
    ):
        rows.append(
            {
                "dataset": dataset_name,
                "variant": variant,
                "fold": fold,
                "subject": subject,
                "accuracy": accuracy_score(
                    sd["true_class"],
                    sd["pred_class"],
                ),
                "balanced_accuracy": balanced_accuracy_score(
                    sd["true_class"],
                    sd["pred_class"],
                ),
                "macro_f1": f1_score(
                    sd["true_class"],
                    sd["pred_class"],
                    average="macro",
                    zero_division=0,
                ),
                "n_trials": len(sd),
            }
        )

    del model, loader, normalizer
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    return pd.DataFrame(rows)


subject_metric_frames = []

if not results_df.empty:
    # Keep diagnostic cost bounded but complete for the confirmation run.
    for _, row in results_df.iterrows():
        subject_metric_frames.append(
            compute_subject_metrics_for_checkpoint(
                row
            )
        )

subject_metrics_df = (
    pd.concat(
        subject_metric_frames,
        ignore_index=True,
    )
    if subject_metric_frames
    else pd.DataFrame()
)

if not subject_metrics_df.empty:
    display(
        subject_metrics_df.sort_values(
            [
                "dataset",
                "variant",
                "fold",
                "accuracy",
            ]
        ).head(40)
    )


In [ ]:
# ============================================================
# CELL 16 — VARIANT SUMMARY + SELECTION
# ============================================================

assert not results_df.empty

summary_df = (
    results_df
    .groupby(
        ["dataset", "variant"],
        as_index=False,
    )
    .agg(
        folds=("fold", "count"),
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        std_balanced_accuracy=(
            "balanced_accuracy",
            "std",
        ),
        mean_macro_f1=("macro_f1", "mean"),
        std_macro_f1=("macro_f1", "std"),
        mean_kappa=("kappa", "mean"),
    )
)

summary_df = summary_df.fillna(0.0)

print("=" * 78)
print("MODULE 11 VARIANT SUMMARY")
print("=" * 78)
display(summary_df)

# Dataset-balanced global score:
# first average across datasets, then average equally across datasets.
dataset_variant_scores = (
    summary_df
    .groupby(
        "variant",
        as_index=False,
    )
    .agg(
        global_balanced_accuracy=(
            "mean_balanced_accuracy",
            "mean",
        ),
        global_macro_f1=(
            "mean_macro_f1",
            "mean",
        ),
        global_std_balanced_accuracy=(
            "std_balanced_accuracy",
            "mean",
        ),
    )
)

dataset_variant_scores["selection_key"] = list(
    zip(
        dataset_variant_scores[
            "global_balanced_accuracy"
        ],
        dataset_variant_scores[
            "global_macro_f1"
        ],
        -dataset_variant_scores[
            "global_std_balanced_accuracy"
        ],
    )
)

winner_row = max(
    dataset_variant_scores.to_dict(
        "records"
    ),
    key=lambda r: r["selection_key"],
)

SELECTED_VARIANT = winner_row["variant"]

print()
print(
    "SOURCE-ONLY SELECTED VARIANT:",
    SELECTED_VARIANT,
)
print(
    "Selection:",
    "mean BA across datasets → "
    "mean Macro-F1 → lower mean SD"
)

display(
    dataset_variant_scores.sort_values(
        [
            "global_balanced_accuracy",
            "global_macro_f1",
        ],
        ascending=False,
    )
)


In [ ]:
# ============================================================
# CELL 17 — CONFUSION MATRICES + PER-CLASS SUMMARY
# ============================================================

from sklearn.metrics import classification_report

cm_rows = []

for _, row in results_df.iterrows():

    dataset_name = row["dataset"]
    variant = row["variant"]
    fold = int(row["fold"])

    d = meta.loc[
        meta["dataset"] == dataset_name
    ].copy()

    splits = list(
        GroupKFold(
            n_splits=CFG.n_splits
        ).split(
            d.index.to_numpy(),
            d["class_id"].to_numpy(),
            d["subject"].to_numpy(),
        )
    )

    train_pos, val_pos = splits[fold - 1]

    idx = d.index.to_numpy()

    val_idx = idx[val_pos]

    train_idx = idx[train_pos]

    train_df = meta.loc[train_idx]
    val_df = meta.loc[val_idx]

    normalizer = SourceRobustNormalizer()
    normalizer.fit(
        X_all[train_idx],
        train_df["subject"].to_numpy(),
        dataset_name,
    )
    normalizer.assert_target_excluded(
        val_df["subject"].to_numpy()
    )

    loader = make_loader(
        normalizer.transform(X_all[val_idx]),
        val_df["class_id"].to_numpy(),
        val_df["subject"].to_numpy(),
        training=False,
    )

    model = FrozenVariantABackbone().to(DEVICE)

    checkpoint = torch.load(
        row["checkpoint"],
        map_location="cpu",
    )

    model.load_state_dict(
        checkpoint["model_state"]
    )
    model.eval()

    ys, ps = [], []

    with torch.no_grad():
        for batch in loader:
            x, y, _ = move_batch_to_device(batch)
            logits = model(x)["logits"]
            ys.extend(
                y.cpu().numpy().tolist()
            )
            ps.extend(
                logits.argmax(
                    dim=1
                ).cpu().numpy().tolist()
            )

    report = classification_report(
        ys,
        ps,
        labels=list(range(len(CFG.classes))),
        target_names=list(CFG.classes),
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        ys,
        ps,
        labels=list(range(len(CFG.classes))),
    )

    cm_rows.append(
        {
            "dataset": dataset_name,
            "variant": variant,
            "fold": fold,
            "confusion_matrix": cm.tolist(),
            "left_recall": report["left"]["recall"],
            "right_recall": report["right"]["recall"],
            "feet_recall": report["feet"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"],
        }
    )

    del model, loader, normalizer
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

cm_df = pd.DataFrame(cm_rows)

display(cm_df)


In [ ]:
# ============================================================
# CELL 18 — LEAKAGE + PROTOCOL ASSERTIONS
# ============================================================

# 1. Every result must be grouped by subject.
assert (
    results_df["n_train_subjects"]
    > 0
).all()
assert (
    results_df["n_val_subjects"]
    > 0
).all()

# 2. Explicit train/validation subject disjointness.
for _, row in results_df.iterrows():
    train_subjects = set(
        row["train_subjects"].split("|")
        if row["train_subjects"]
        else []
    )
    val_subjects = set(
        row["val_subjects"].split("|")
        if row["val_subjects"]
        else []
    )

    assert train_subjects.isdisjoint(
        val_subjects
    )

# 3. Normalizer is fitted on training subjects only.
for _, row in results_df.iterrows():
    fitted = set(
        row["normalizer_subjects"].split("|")
        if row["normalizer_subjects"]
        else []
    )
    val = set(
        row["val_subjects"].split("|")
        if row["val_subjects"]
        else []
    )

    assert fitted.isdisjoint(val)
    assert bool(row["target_excluded"])

# 4. No target dataset enters selection.
assert set(
    summary_df["dataset"].unique()
) == set(dataset_names)

# 5. No subject deletion.
assert set(
    meta["subject"].unique()
) == set(meta["subject"].unique())

# 6. Parameter/shape expectations.
check_model = FrozenVariantABackbone()
assert check_model.classifier.out_features == 3
assert (
    check_model.embedding[-3].normalized_shape
    == (CFG.embedding_dim,)
    if isinstance(
        check_model.embedding[-3],
        nn.LayerNorm,
    )
    else True
)
del check_model

print("=" * 78)
print("MODULE 11 LEAKAGE / PROTOCOL AUDIT")
print("=" * 78)
print("Grouped split                : PASS")
print("Normalizer target exclusion  : PASS")
print("Target metrics used          : NO")
print("Target tuning                : NO")
print("Target augmentation          : NO")
print("Subject deletion             : NO")
print("Source-only variant selection: PASS")


In [ ]:
# ============================================================
# CELL 19 — STATISTICAL COMPARISON OF VARIANTS
# ============================================================

# Fold-level paired comparison within each dataset.
# Nonparametric Wilcoxon is used only when >= 3 paired folds exist.

try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

stat_rows = []

if SCIPY_AVAILABLE:
    for dataset_name in dataset_names:
        pivot = (
            results_df.loc[
                results_df["dataset"]
                == dataset_name
            ]
            .pivot(
                index="fold",
                columns="variant",
                values="balanced_accuracy",
            )
        )

        for variant in CFG.run_variants:
            if variant == "A0":
                continue

            if (
                "A0" not in pivot.columns
                or variant not in pivot.columns
            ):
                continue

            paired = pivot[
                ["A0", variant]
            ].dropna()

            if len(paired) >= 3:
                try:
                    stat, p = wilcoxon(
                        paired["A0"],
                        paired[variant],
                        zero_method="wilcox",
                        alternative="two-sided",
                    )
                except Exception:
                    stat, p = np.nan, np.nan
            else:
                stat, p = np.nan, np.nan

            stat_rows.append(
                {
                    "dataset": dataset_name,
                    "comparison": f"A0 vs {variant}",
                    "n_paired_folds": len(paired),
                    "wilcoxon_stat": stat,
                    "p_value": p,
                }
            )

stat_df = pd.DataFrame(stat_rows)

if not stat_df.empty:
    display(stat_df)
else:
    print(
        "No paired statistical test available."
    )


In [ ]:
# ============================================================
# CELL 20 — FINAL MODULE 11 REPORT
# ============================================================

print("=" * 78)
print("MODULE 11 v1 — FINAL SOURCE-ONLY CONFIRMATION")
print("=" * 78)

print(
    f"Datasets confirmed: {', '.join(dataset_names)}"
)
print(
    f"Grouped folds per dataset: "
    f"{CFG.n_splits if not CFG.smoke_test else CFG.smoke_folds_per_dataset}"
)

print()
print("Selected variant:", SELECTED_VARIANT)

selected_scores = dataset_variant_scores.loc[
    dataset_variant_scores["variant"]
    == SELECTED_VARIANT
].iloc[0]

print(
    "Global mean balanced accuracy : "
    f"{selected_scores['global_balanced_accuracy']:.4f}"
)
print(
    "Global mean macro-F1          : "
    f"{selected_scores['global_macro_f1']:.4f}"
)

print()
print("Variant comparison:")
display(
    summary_df.sort_values(
        [
            "dataset",
            "mean_balanced_accuracy",
        ],
        ascending=[True, False],
    )
)

print()
print("=" * 78)
print("PROTOCOL STATUS")
print("=" * 78)

status = {
    "backbone_fixed_variant_A": True,
    "three_fold_grouped_confirmation": (
        CFG.n_splits == 3
        or CFG.smoke_test
    ),
    "both_source_datasets_confirmed": (
        set(results_df["dataset"].unique())
        == set(dataset_names)
    ),
    "target_subjects_used": False,
    "target_normalization_used": False,
    "target_tuning_used": False,
    "target_augmentation_used": False,
    "subject_deletion_used": False,
    "variant_selected_source_only": True,
    "checkpoints_saved": True,
    "history_saved": True,
}

for key, value in status.items():
    print(
        f"{key:38s}: "
        f"{'PASS' if value else 'FAIL'}"
    )

assert all(status.values()), (
    "Module 11 protocol gate failed."
)

print()
print("MODULE 11 STATUS: PASS")


In [ ]:
# ============================================================
# CELL 21 — SAVE RESULTS + FREEZE MANIFEST
# ============================================================

RESULTS_PATH = (
    MODULE11_ROOT
    / "module_11_fold_results.csv"
)

SUMMARY_PATH = (
    MODULE11_ROOT
    / "module_11_variant_summary.csv"
)

SUBJECT_RESULTS_PATH = (
    MODULE11_ROOT
    / "module_11_subject_metrics.csv"
)

CM_PATH = (
    MODULE11_ROOT
    / "module_11_confusion_summary.csv"
)

SPEC_PATH = (
    MODULE11_ROOT
    / "module_11_specification.json"
)

FREEZE_PATH = (
    MODULE11_ROOT
    / "module_11_source_only_freeze_manifest.json"
)

results_df.to_csv(
    RESULTS_PATH,
    index=False,
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)

if not subject_metrics_df.empty:
    subject_metrics_df.to_csv(
        SUBJECT_RESULTS_PATH,
        index=False,
    )

cm_df.to_csv(
    CM_PATH,
    index=False,
)

spec = {
    "module": 11,
    "version": "v1",
    "name": "Controlled Subject/Domain Alignment",
    "backbone": "Module 10A Variant A",
    "datasets": list(dataset_names),
    "classes": list(CFG.classes),
    "input": {
        "channels": CFG.n_channels,
        "samples": CFG.n_samples,
        "sampling_rate_hz": CFG.target_sfreq,
        "band_hz": [
            CFG.low_hz,
            CFG.high_hz,
        ],
    },
    "variants": {
        "A0": "CE only",
        "A1": "CE + batch-level mean-statistics subject alignment",
        "A2": "CE + subject-adversarial GRL",
        "A3": "CE + class-conditional subject alignment",
    },
    "selection_rule": [
        "mean balanced accuracy across source datasets",
        "mean macro F1",
        "lower mean balanced-accuracy SD",
    ],
    "protocol": {
        "group_column": "subject",
        "grouped_validation_folds": CFG.n_splits,
        "target_subjects_used": False,
        "target_normalization_used": False,
        "target_tuning_used": False,
        "target_augmentation_used": False,
        "target_pseudolabeling_used": False,
        "subject_deletion": False,
        "source_only_selection": True,
    },
    "selected_variant": SELECTED_VARIANT,
    "output_paths": {
        "fold_results": str(RESULTS_PATH),
        "variant_summary": str(SUMMARY_PATH),
        "subject_metrics": str(
            SUBJECT_RESULTS_PATH
        ),
        "confusion_summary": str(CM_PATH),
        "checkpoints": str(CHECKPOINT_ROOT),
        "histories": str(HISTORY_ROOT),
    },
}

with open(
    SPEC_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        spec,
        f,
        indent=2,
    )

freeze_manifest = {
    "module": 11,
    "version": "v1",
    "status": "LOCKED_FOR_NEXT_MODULE",
    "selected_variant": SELECTED_VARIANT,
    "selection_basis": (
        "source-only 3-fold grouped confirmation "
        "on BCI-IV-2a and EEGMMIDB"
    ),
    "target_subjects_used": False,
    "target_metrics_used": False,
    "target_normalization_used": False,
    "target_tuning_used": False,
    "target_augmentation_used": False,
    "low_performing_subjects_removed": False,
    "backbone": (
        "Frozen Module 10A Variant A"
    ),
    "next_step": (
        "Module 12 — contrastive/domain-generalization "
        "study using the source-only selected strategy"
    ),
}

with open(
    FREEZE_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        freeze_manifest,
        f,
        indent=2,
    )

print("=" * 78)
print("MODULE 11 ARTIFACTS SAVED")
print("=" * 78)
print("Fold results :", RESULTS_PATH)
print("Summary      :", SUMMARY_PATH)
print("Subject      :", SUBJECT_RESULTS_PATH)
print("Confusion    :", CM_PATH)
print("Specification:", SPEC_PATH)
print("Freeze       :", FREEZE_PATH)


## Execution guidance

### First run
Run **Cells 1 → 21 sequentially** with:

```python
CFG.smoke_test = True
```

inside Cell 2 for a quick engineering check.

### Confirmation run
After the smoke run passes, set:

```python
CFG.smoke_test = False
CFG.n_splits = 3
```

and rerun **Cells 2 → 21**.

### Interpretation
Do not select a variant based on an individual strong subject. The primary selection metric is mean balanced accuracy across the source datasets, followed by macro-F1 and stability.

Do not remove difficult subjects. They remain in the confirmation set and are included in the primary analysis.

### Important
A high Module 11 development score is **not** a final LOSO or cross-dataset result. The selected strategy must still pass the subsequent outer evaluation protocol before it can support a final scientific claim.
